In [1]:
import os
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

RANDOM_STATE = 42

DATA_DIR = "/kaggle/input/petfinder-pawpularity-score"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TEST_CSV = os.path.join(DATA_DIR, "test.csv")

df_train = pd.read_csv(TRAIN_CSV)
df_test = pd.read_csv(TEST_CSV)

metadata_features = [
    "Subject Focus",
    "Eyes",
    "Face",
    "Near",
    "Action",
    "Accessory",
    "Group",
    "Collage",
    "Human",
    "Occlusion",
    "Info",
    "Blur",
]

missing_train = [
    c for c in (metadata_features + ["Pawpularity"]) if c not in df_train.columns
]
missing_test = [c for c in metadata_features if c not in df_test.columns]
if missing_train:
    raise ValueError(f"Missing columns in train.csv: {missing_train}")
if missing_test:
    raise ValueError(f"Missing columns in test.csv: {missing_test}")

X = df_train[metadata_features].to_numpy(dtype=np.float32)
y = df_train["Pawpularity"].to_numpy(dtype=np.float32)
X_test = df_test[metadata_features].to_numpy(dtype=np.float32)

# Change (score + validity): add a fixed train/valid split so we can use early stopping
# to reduce overfitting and generally improve RMSE vs training on all data unchecked.
X_tr, X_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE
)



In [2]:
model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="reg:squarederror",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)

# Change (score): early stopping keeps the same core model/metric/objective,
# but typically improves generalization RMSE for this baseline.
model.fit(
    X_tr,
    y_tr,
    eval_set=[(X_va, y_va)],
    eval_metric="rmse",
    verbose=False,
    early_stopping_rounds=100,
)

# Optional sanity: report validation RMSE (not used for submission).
va_pred = model.predict(X_va)
va_rmse = float(mean_squared_error(y_va, va_pred, squared=False))
print("Validation RMSE:", va_rmse)

preds = model.predict(X_test)
preds = np.clip(preds, 0.0, 100.0)

submission = pd.DataFrame(
    {"Id": df_test["Id"], "Pawpularity": preds.astype(np.float32)}
)

# Change (validity): ensure row alignment is correct; prevents invalid submissions.
if submission.shape[0] != df_test.shape[0]:
    raise RuntimeError(
        f"Submission row count {submission.shape[0]} does not match test row count {df_test.shape[0]}"
    )

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Wrote submission.csv with shape:", submission.shape)

Validation RMSE: 21.363595962524414
                                 Id  Pawpularity
0  ee51b99832f1ba868f646df93d2b6b81    37.936321
1  caddfb3f8bff9c4b95dbe022018eea21    41.681377
2  582eeabd4a448a53ebb79995888a4b0b    37.573750
3  afc1ad7f0c5eea880759d09e77f7deee    37.936321
4  d5bdf3446e86ce4ec67ce7a00f1cccc2    34.795322
Wrote submission.csv with shape: (992, 2)


/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
